# Data Preparation - NER Pipeline ve CoNLL Oluşturma

Bu notebook, healthcare dataset'lerini yükler, NER pipeline'ını çalıştırır, entity'leri çıkarır ve CoNLL formatında kaydeder.

## Adımlar:
1. **Setup & License** - Spark NLP Healthcare lisansı ve ortam kurulumu
2. **Dataset Yükleme** - Healthcare dataset'lerini yükleme ve hazırlama
3. **NER Pipeline** - Pre-trained modellerle NER pipeline çalıştırma
4. **Entity Extraction** - Entity'leri çıkarma ve birleştirme (Priority: Posology > DeID > Clinical)
5. **CoNLL Oluşturma** - Entity'leri CoNLL formatına dönüştürme

**Çıktılar:**
- `data/processed/text_data.csv` - Hazırlanmış text verisi
- `data/processed/entities.csv` - Çıkarılan entity'ler
- `data/conll/conll2003_text_file.conll` - CoNLL formatında training verisi


## 1. Setup & License Configuration


In [1]:
import json
import os
from pathlib import Path

# License dosyasını yükle (Kaggle/Colab için)
# Kaggle için:
try:
    with open(r'C:\huseyin\john_snow_labs\generating_conll_files_from_pretrained_models\spark_jsl.json') as f:
        license_keys = json.load(f)
except:
    # Colab için:
    try:
        from google.colab import files
        if 'spark_jsl.json' not in os.listdir():
            print("Please upload your spark_jsl.json license file:")
            uploaded = files.upload()
            os.rename(list(uploaded.keys())[0], 'spark_jsl.json')
        with open('spark_jsl.json') as f:
            license_keys = json.load(f)
    except:
        # Local için:
        with open('spark_jsl.json') as f:
            license_keys = json.load(f)

# License key'leri environment variable olarak ayarla
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.1.1
Public Version: 6.1.3


In [2]:
# Java ve kütüphaneleri yükle
import subprocess
import sys

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java: {java_version.split(chr(10))[0]}")
except:
    print("⚠️ Java bulunamadı, yükleniyor...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
has_gpu = gpu_check.returncode == 0

if has_gpu:
    print("🚀 GPU detected! Installing PyTorch with CUDA...")
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("Installing PyTorch (CPU version)...")
    !pip install -q torch torchvision torchaudio

# Spark NLP ve diğer kütüphaneler
!pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION
!pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET
!pip install -q spark-nlp-display pandas numpy tqdm requests

print("✅ All libraries installed")


⚠️ Java bulunamadı, yükleniyor...
🚀 GPU detected! Installing PyTorch with CUDA...
✅ All libraries installed


In [ ]:
# Spark Session başlat
import sparknlp
import sparknlp_jsl
from pyspark.sql import SparkSession
import os
from pathlib import Path

# GPU kontrolü
try:
    import torch
    gpu_available = torch.cuda.is_available()
except:
    gpu_available = False

# Lokal ortam için cache klasörleri
current_dir = Path(os.getcwd())
cache_dir = current_dir.parent / "cache_pretrained"
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = str(cache_dir.absolute())

# Spark konfigürasyonu - lokal ortam için optimize edilmiş
params = {
    "spark.driver.memory": "8G",  # Lokal için daha düşük
    "spark.kryoserializer.buffer.max": "1000M",
    "spark.driver.maxResultSize": "1000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.driver.extraJavaOptions": "-Xmx6g"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": cache_path,
        "spark.jsl.settings.storage.cluster_tmp_dir": cache_path,
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled")
else:
    # CPU için cache ayarları
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": cache_path,
        "spark.jsl.settings.storage.cluster_tmp_dir": cache_path
    })

# Spark session başlat - hata yakalama ile
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")  # Log seviyesini düşür
    
    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized")
    
    # Test et
    test_df = spark.createDataFrame([("test",)], ["text"])
    test_df.show(truncate=False)
    print("✅ Spark session is working correctly")
    
except Exception as e:
    print(f"❌ Error starting Spark session with sparknlp_jsl.start(): {e}")
    print("\nTrying alternative method with SparkSession.builder...")
    
    # Alternatif yöntem: SparkSession.builder kullan
    try:
        builder = (
            SparkSession.builder
            .appName("SparkNLPHealthcare")
            .config("spark.driver.memory", "8G")
            .config("spark.kryoserializer.buffer.max", "1000M")
            .config("spark.driver.maxResultSize", "1000M")
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
            .config("spark.driver.extraJavaOptions", "-Xmx6g")
        )
        
        if gpu_available:
            builder = builder.config("spark.jsl.settings.annotator.gpu", "true")
            print("✅ GPU acceleration enabled (alternative method)")
        
        spark = builder.getOrCreate()
        spark.sparkContext.setLogLevel("ERROR")
        
        # sparknlp_jsl'i manuel olarak yapılandır
        spark.conf.set("spark.jsl.settings.pretrained.cache_folder", cache_path)
        spark.conf.set("spark.jsl.settings.storage.cluster_tmp_dir", cache_path)
        
        print(f"✅ Spark NLP Version: {sparknlp.version()}")
        print("✅ Spark session initialized (alternative method)")
        
        # Test et
        test_df = spark.createDataFrame([("test",)], ["text"])
        test_df.show(truncate=False)
        print("✅ Spark session is working correctly")
        
    except Exception as e2:
        print(f"❌ Failed with alternative method: {e2}")
        print("\n⚠️ Please check:")
        print("   1. Java is installed (java -version)")
        print("   2. JAVA_HOME environment variable is set")
        print("   3. Spark NLP JSL packages are properly installed")
        raise


✅ GPU acceleration enabled
Starting Spark session...


In [ ]:
# Modülleri import et
import sys
import os
from pathlib import Path

# Notebook'un bulunduğu dizini tespit et
# Jupyter notebook'larda os.getcwd() genellikle notebook'un bulunduğu dizini verir
current_dir = Path(os.getcwd())

# src dizinini bul - önce lokal ortam için, sonra Kaggle/Colab için
src_paths = [
    # Lokal ortam: notebooks/ klasöründen bir üst dizindeki src
    current_dir.parent / 'src',
    # Lokal ortam: mevcut dizindeki src (eğer notebook root'ta çalışıyorsa)
    current_dir / 'src',
    # Kaggle/Colab ortamları
    Path('/kaggle/working/src'),
    Path('/content/src'),
    # Relative paths
    Path('../src'),
    Path('src'),
]

src_path = None
for path in src_paths:
    if path.exists() and path.is_dir():
        # src klasörünün parent'ını path'e ekle (src/__init__.py için)
        if str(path.parent) not in sys.path:
            sys.path.insert(0, str(path.parent))
        src_path = path
        print(f"✅ Found src directory at: {path}")
        break

if not src_path:
    print("⚠️ src directory not found in any of the following locations:")
    for path in src_paths:
        print(f"   - {path}")
    print("\nPlease ensure the src directory exists relative to the notebook location.")
else:
    # Modülleri import et
    try:
        from src import (
            DatasetLoader,
            NERPipeline,
            CoNLLConverter,
            extract_entities_from_ner_results,
            get_entity_statistics
        )
        print("✅ All modules imported successfully")
    except ImportError as e:
        print(f"❌ Import error: {e}")
        print(f"   Make sure src/__init__.py exists and exports all required modules")


## 2. Dataset Yükleme


In [ ]:
# Dataset loader'ı başlat
data_dir = "data/raw"
loader = DatasetLoader(data_dir=data_dir)

# mtsamples_classifier dataset'ini indir
print("Downloading dataset...")
df = loader.download_mtsamples_classifier()
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


In [ ]:
# Text dataframe'i hazırla
text_df = loader.prepare_text_dataframe(df, text_column="text", id_column=None)

# Kaydet
output_path = Path("data/processed/text_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
text_df.to_csv(output_path, index=False)

print(f"✅ Saved {len(text_df)} texts to {output_path}")
text_df.head()


## 3. NER Pipeline Çalıştırma


In [ ]:
# NER Pipeline oluştur
ner_pipeline = NERPipeline(spark, license_secret=license_keys['SECRET'])
pipeline = ner_pipeline.create_pipeline(prioritize_posology_deid=True)

print("✅ NER pipeline created")
print("Models in pipeline:")
print("  - ner_clinical")
print("  - ner_deid_generic_augmented")
print("  - ner_posology (Drug, Dosage only)")


In [ ]:
# Spark DataFrame'e dönüştür
spark_df = spark.createDataFrame(text_df)
print(f"Spark DataFrame created with {spark_df.count()} rows")
spark_df.show(5, truncate=100)


In [ ]:
# NER pipeline'ını çalıştır
print("Running NER pipeline... This may take several minutes...")
result_df = ner_pipeline.fit_transform(spark_df)

print("✅ NER pipeline completed")
result_df.select("text_id", "text", "chunk_clinical", "chunk_deid", "chunk_posology").show(5, truncate=100)


## 4. Entity Extraction ve Birleştirme


In [ ]:
# Entity'leri çıkar ve birleştir (Priority: Posology > DeID > Clinical)
print("Extracting entities from NER results...")
entity_df = extract_entities_from_ner_results(result_df, text_df)

print(f"✅ Extracted {len(entity_df)} entities")
print(f"\nEntity types: {sorted(entity_df['entity'].unique())}")
print(f"\nEntity distribution:")
print(entity_df['entity'].value_counts())
entity_df.head(10)


In [ ]:
# Entity istatistikleri
stats = get_entity_statistics(entity_df)
print("Entity Statistics:")
print(f"  Total entities: {stats['total_entities']}")
print(f"  Unique entity types: {stats['unique_entity_types']}")
print(f"  Texts with entities: {stats['texts_with_entities']}")
print(f"  Avg entities per text: {stats['avg_entities_per_text']:.2f}")

# Entity'leri kaydet
entity_output_path = Path("data/processed/entities.csv")
entity_output_path.parent.mkdir(parents=True, exist_ok=True)
entity_df.to_csv(entity_output_path, index=False)
print(f"\n✅ Saved entities to {entity_output_path}")


## 5. CoNLL Formatına Dönüştürme


In [ ]:
# CoNLL converter'ı başlat
conll_converter = CoNLLConverter(spark)

# CoNLL dosyası oluştur
conll_path = "data/conll/conll2003_text_file.conll"
print("Creating CoNLL file...")
conll_text = conll_converter.make_conll(
    text_df=text_df,
    entity_df=entity_df,
    save_tag=True,
    save_conll=True,
    output_path=conll_path,
    verbose=False
)

print(f"\n✅ CoNLL file created: {conll_path}")
print(f"CoNLL file size: {len(conll_text)} characters")


## Özet

✅ **Data preparation tamamlandı!**

**Oluşturulan dosyalar:**
- `data/processed/text_data.csv` - Hazırlanmış text verisi
- `data/processed/entities.csv` - Çıkarılan entity'ler
- `data/conll/conll2003_text_file.conll` - CoNLL formatında training verisi

**Sonraki adım:** `2_model_training.ipynb` notebook'unu çalıştırarak model eğitimi yapabilirsiniz.
